# Point Cloud Viewer

Interactive notebook for viewing and comparing point cloud data (LAS, LAZ, PLY formats).

In [29]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

# Point cloud libraries
import laspy
from plyfile import PlyData

In [30]:
def read_ply_file(file_path):
    """Read PLY file"""
    plydata = PlyData.read(str(file_path))
    vertex_data = plydata['vertex']
    return {
        'x': vertex_data.data['x'],
        'y': vertex_data.data['y'],
        'z': vertex_data.data['z'],
        'raw_data': vertex_data.data,
        'properties': vertex_data.data.dtype.names,
        'num_points': len(vertex_data.data),
        'file_type': 'PLY'
    }

def read_las_file(file_path):
    """Read LAS/LAZ file"""
    las_file = laspy.read(str(file_path))
    return {
        'x': las_file.x,
        'y': las_file.y,
        'z': las_file.z,
        'raw_data': las_file,
        'properties': list(las_file.point_format.dimension_names),
        'num_points': len(las_file.points),
        'file_type': 'LAS/LAZ',
        'header': las_file.header
    }

def read_point_cloud(file_path):
    """Read point cloud from PLY, LAS, or LAZ file"""
    file_path = Path(file_path)
    ext = file_path.suffix.lower()

    if ext == '.ply':
        return read_ply_file(file_path)
    elif ext in ['.las', '.laz']:
        return read_las_file(file_path)
    else:
        raise ValueError(f"Unsupported format: {ext}")
def display_summary(data, name="Point Cloud"):
    """Display a summary table for a point cloud"""
    display(Markdown(f"### {name}"))

    summary = pd.DataFrame({
        'Property': ['File Type', 'Number of Points', 'Available Fields'],
        'Value': [
            data['file_type'],
            f"{data['num_points']:,}",
            ', '.join(data['properties'])
        ]
    })
    display(summary.style.hide(axis='index'))

def display_coordinate_ranges(data):
    """Display X/Y/Z ranges (Min / Max / Range / Unique).

    Thin wrapper around the general `display_field_ranges` so coordinates and any
    other columns are summarised the same way.
    """
    display_field_ranges(data, ['x', 'y', 'z'], title="Coordinate Ranges")

def _has_field(data, field):
    """Return True if `field` is available on this point cloud."""
    if data['file_type'] == 'PLY':
        return field in data['raw_data'].dtype.names
    # LAS/LAZ: works for x/y/z and every (incl. extra) dimension, e.g. tree_ID, dist_axes
    return hasattr(data['raw_data'], field)

def _get_field(data, field, n):
    """Return the first `n` values of `field` as a plain 1-D numpy array.

    np.asarray() is essential: laspy exposes x/y/z as ScaledArrayView and bit fields
    (e.g. return_number) as SubFieldView. pandas does not treat those views as
    array-like and would broadcast the whole view into every row, so we materialise
    them into real ndarrays here.
    """
    if data['file_type'] == 'PLY':
        return np.asarray(data['raw_data'][field][:n])
    return np.asarray(getattr(data['raw_data'], field)[:n])

def _get_column(data, field):
    """Return the full column for `field` as a plain 1-D numpy array (all points)."""
    if data['file_type'] == 'PLY':
        return np.asarray(data['raw_data'][field])
    return np.asarray(getattr(data['raw_data'], field))

def display_sample_points(data, n=10, fields=None):
    """Display the first N points, showing only the requested `fields` (columns).

    `fields` is a list of column names to display, e.g. ['x', 'y', 'z', 'tree_ID'].
    Names the file does not contain are skipped with a warning. When `fields` is
    None, every available field is shown.
    """
    n = min(n, data['num_points'])

    if fields is None:
        selected = list(data['properties'])
    else:
        selected = [f for f in fields if _has_field(data, f)]
        missing = [f for f in fields if not _has_field(data, f)]
        if missing:
            display(Markdown(
                f"> ⚠️ Skipping field(s) not in this file: `{', '.join(map(str, missing))}`  \n"
                f"> Available: `{', '.join(map(str, data['properties']))}`"
            ))

    if not selected:
        display(Markdown("**First points:** _no matching fields to display._"))
        return

    df = pd.DataFrame({f: _get_field(data, f, n) for f in selected})

    display(Markdown(f"**First {n} Points:**"))
    display(df)

def display_field_ranges(data, fields, title="Field Ranges"):
    """Display Min / Max / Range / number of Unique values for any numeric column(s).

    Generalises `display_coordinate_ranges` to arbitrary fields, e.g.
    `display_field_ranges(data, ['tree_ID'], title="TreeID Ranges")` or
    `display_field_ranges(data, ['dist_axes', 'Z0'])`. Fields the file does not
    contain, or non-numeric fields, are skipped.
    """
    rows = []
    for f in fields:
        if not _has_field(data, f):
            continue
        col = _get_column(data, f)
        if not np.issubdtype(col.dtype, np.number):
            continue  # ranges only make sense for numeric columns
        cmin, cmax = col.min(), col.max()
        n_unique = np.unique(col).size
        rows.append({
            'Field': f,
            'Min': f"{cmin:.3f}",
            'Max': f"{cmax:.3f}",
            'Range': f"{cmax - cmin:.3f}",
            'Unique': f"{n_unique:,}",
        })

    if not rows:
        display(Markdown(f"**{title}:** _no matching numeric fields._"))
        return

    display(Markdown(f"**{title}:**"))
    display(pd.DataFrame(rows).style.hide(axis='index'))

def count_unique(data, field):
    """Return the number of distinct values in `field` (over all points)."""
    return int(np.unique(_get_column(data, field)).size)

def display_unique_counts(data, fields, title="Unique Value Counts"):
    """Display the number of distinct values in each of `fields`.

    Suited to categorical columns such as `tree_ID`, where the count of unique
    values is the useful statistic (e.g. how many individual trees a plot holds)
    while Min/Max/Range are not meaningful. Missing fields are skipped.
    """
    rows = []
    for f in fields:
        if not _has_field(data, f):
            continue
        rows.append({'Field': f, 'Unique values': f"{count_unique(data, f):,}"})

    if not rows:
        display(Markdown(f"**{title}:** _no matching fields._"))
        return

    display(Markdown(f"**{title}:**"))
    display(pd.DataFrame(rows).style.hide(axis='index'))

def to_dataframe(data, fields=None, n=None):
    """Return the point cloud as a pandas DataFrame to sort / rename / re-value columns.

    fields : list of columns to include (default: every available field; unknown names
             are skipped).
    n      : keep only the first n points (default: None = all points).

    Note: with n=None a large cloud materialises every requested column in memory
    (e.g. ~18M rows). Pass a small n, or a short `fields` list, to preview.

    Example:
        df = to_dataframe(my_data, fields=['x', 'y', 'z', 'tree_ID'])
        df = df.sort_values('tree_ID')                    # sort
        df = df.rename(columns={'tree_ID': 'tree'})       # rename
        df['tree'] = df['tree'].replace({0: -1})          # re-value
    """
    if fields is None:
        fields = list(data['properties'])
    else:
        fields = [f for f in fields if _has_field(data, f)]

    cols = {}
    for f in fields:
        col = _get_column(data, f)
        cols[f] = col if n is None else col[:n]
    return pd.DataFrame(cols)

def view_point_cloud(file_path, fields=None, range_fields=None, range_title="Field Ranges"):
    """Full view of a single point cloud file.

    fields       : columns shown in the "First N Points" table (None = all fields).
    range_fields : columns summarised as Min/Max/Range under `range_title`
                   (None = skip; e.g. ['tree_ID'] adds a "TreeID Ranges" section).
    """
    file_path = Path(file_path)
    data = read_point_cloud(file_path)

    display_summary(data, name=file_path.name)
    display_coordinate_ranges(data)
    display_sample_points(data, fields=fields)
    if range_fields:
        display_field_ranges(data, range_fields, title=range_title)

    return data
def compare_files(file_paths, names=None):
    """Compare multiple point cloud files side by side"""
    if names is None:
        names = [Path(p).name for p in file_paths]

    data_list = [read_point_cloud(p) for p in file_paths]

    # Build comparison table
    comparison = {
        'File': names,
        'Points': [f"{d['num_points']:,}" for d in data_list],
        'X Min': [f"{d['x'].min():.3f}" for d in data_list],
        'X Max': [f"{d['x'].max():.3f}" for d in data_list],
        'Y Min': [f"{d['y'].min():.3f}" for d in data_list],
        'Y Max': [f"{d['y'].max():.3f}" for d in data_list],
        'Z Min': [f"{d['z'].min():.3f}" for d in data_list],
        'Z Max': [f"{d['z'].max():.3f}" for d in data_list],
    }

    display(Markdown("## File Comparison"))
    display(pd.DataFrame(comparison))

    return data_list

---
## Load Your Own File

Modify the path below to view any point cloud file.

In [31]:
# Columns to show in the "First 10 Points" table — edit this list.
# Unknown names are skipped automatically; check the "Available Fields" summary
# (or my_data['properties']) for what a given file offers. For the 3DFin tree files,
# the useful extra dimensions are 'tree_ID' and 'dist_axes'.
fields_to_display = ['x', 'y', 'z', 'tree_ID']

# Modify the path to view your own file:
my_data = view_point_cloud(
    r"D:\Podyplomowe\04_AI_Intro\assignment\SegmentedForests\3DFin_output\plot_01\plot_01_tree_ID_dist_axes.las",
    fields=fields_to_display,
    range_fields=['tree_ID'],       # -> adds the "TreeID Ranges" (Min / Max / Range) section
    range_title="TreeID Ranges",
)

# Number of distinct values in tree_ID (i.e. how many individual trees, incl. the 0 = "no tree" id):
display_unique_counts(my_data, ['tree_ID'])

# ...or grab the plain number for use in code:
print(f"Unique tree_ID values: {count_unique(my_data, 'tree_ID'):,}")


### plot_01_tree_ID_dist_axes.las

Property,Value
File Type,LAS/LAZ
Number of Points,"20,831,953"
Available Fields,"X, Y, Z, intensity, return_number, number_of_returns, synthetic, key_point, withheld, overlap, scanner_channel, scan_direction_flag, edge_of_flight_line, classification, user_data, scan_angle, point_source_id, gps_time, red, green, blue, Class, Split, dist_axes, tree_ID, Z0"


**Coordinate Ranges:**

Field,Min,Max,Range,Unique
x,-15.384,12.724,28.108,"28,063"
y,-12.762,13.760,26.522,"26,507"
z,-4.156,18.167,22.323,"21,472"


**First 10 Points:**

...

**TreeID Ranges:**

Field,Min,Max,Range,Unique
tree_ID,0.000,630619.000,630619.000,119


**Unique Value Counts:**

Field,Unique values
tree_ID,119


Unique tree_ID values: 119


---
## Work with the cloud as a DataFrame

`to_dataframe(...)` returns a plain pandas DataFrame (`.values` / `.to_numpy()` give the
numpy array), so you can **sort**, **rename**, and **re-value** columns with normal pandas.

In [32]:
# Pick the columns you want (fields=None -> all; n=None -> all points).
df = to_dataframe(my_data, fields=['x', 'y', 'z', 'tree_ID', 'Class'])

# Examples — sort / rename / re-value:
df = df.sort_values('tree_ID', ascending=True)        # sort by tree
# df = df.rename(columns={'tree_ID': 'tree'})         # rename a column
# df['tree_ID'] = df['tree_ID'].replace({0: -1})      # re-value (0 = "no tree" -> -1)

df.head(10)


,x,y,z,tree_ID,Class
7028225,8.711000,-9.400,-0.437,0,3
7028221,8.728999,-9.473,-0.106,0,3
7028220,8.766999,-9.344,-0.414,0,3
7028219,8.711000,-9.446,-0.093,0,3
7028218,8.811999,-9.314,-0.402,0,3
7028217,8.763000,-9.548,0.224,0,3
7028216,8.718999,-9.417,-0.080,0,3
7028241,8.778999,-9.475,-0.843,0,3
12662778,7.664999,-9.960,0.124,0,2
12662776,7.697999,-9.982,0.120,0,2
